# PM10 + PM2.5（dyn）S2 Paper Evaluation Pipeline (Notebook)

等价于 `run_paper_eval.ipynb` 的评测逻辑，用于测试 **`PMST_net_test_11_s2_pm10.py`** 在 **动态特征维数 `dyn_vars_count=27`**（在原 PM10 通道基础上增加 **PM2.5**）时训练得到的模型。

**模型与数据变更要点（相对仅 PM10）**
- **输入**：每时间步动态变量从 26 增至 **27**（末两维为 **PM2.5、PM10**，均参与 `log1p` 与 RobustScaler，与训练脚本 `_dyn_indices_log1p` 一致）。
- **输出**：仍为能见度三分类（Fog / Mist / Clear），**`num_classes=3` 未变**。
- **数据目录**：`ml_dataset_s2_tianji_12h_pm10_pm25_monthtail_2`（与训练脚本 `CONFIG['S2_DATA_DIR']` 对齐）。
- **Scaler 文件**：`robust_scaler_w12_dyn27_s2_48h_pm10.pkl`（文件名沿用训练脚本惯例，**必须与当前 dyn 维数一致**）。

**输出（与 `run_paper_eval.py` 对齐）**：`metrics_by_forecast_init.csv`、`fig7b_forecast_init_comparison.png`、`metrics_by_valid_hour.csv`；起报分层需 `meta_test.csv` 含 `init_time`。


In [10]:
## 1. 路径与配置 (PM10 + PM2.5 dyn / 11_s2)

import os
import sys

# 若在 Jupyter 中运行，通常 cwd 为 vis_mlp 或 paper_eval
try:
    _file = __file__
except NameError:
    _file = os.path.join(os.getcwd(), "paper_eval", "run_paper_eval_pm10_pm25_11_s2.ipynb")

BASE = (
    os.path.dirname(os.path.dirname(os.path.abspath(_file)))
    if "ipynb" not in _file
    else os.path.abspath(os.getcwd())
)

# 让相对导入/模块查找更稳定
if not os.path.exists(os.path.join(BASE, "paper_eval")):
    BASE = os.path.dirname(BASE)

PAPER_EVAL_DIR = os.path.join(BASE, "paper_eval")

sys.path.insert(0, BASE)
sys.path.insert(0, os.path.join(BASE, "train"))
os.chdir(BASE)

# ---------- 配置（按需修改） ----------
# 与训练脚本 CONFIG：build_s2_run_exp_id(EXPERIMENT_ID, S2_RUN_SUFFIX) 一致，例如：
# exp_1776227576 + pm10_more_temp_search -> exp_1776227576_pm10_more_temp_search
exp_id = "exp_1776227576_pm10_more_temp_search"
output_dir = "paper_eval_results_pm10_pm25_11_s2"  # None: paper_eval_results_{exp_id}
#output_dir = None
data_dir = os.path.join(BASE, "ml_dataset_s2_tianji_12h_pm10_pm25_monthtail_2")
ckpt_dir = os.path.join(BASE, "checkpoints")

shp_path = "/public/home/putianshu/中华人民共和国/中华人民共和国.shp"
ifs_nc = os.path.join(BASE, "VIS_IDW_KDTree_20250101_20251231.nc")

# 作为“无校准”时的全局阈值；本 notebook 默认启用校准 artifact。
fog_th = 0.10
mist_th = 0.42

no_calibration = True  # False: load season_thresholds.pt + temperature
threshold_rule = "mutual"  # "default" | "mutual" | "joint"

use_cpu = False  # 容器实例无 GPU/或 HIP 报错时改 True
batch_size = 8192

# 大范围雾事件参数（沿用 run_paper_eval）
event_top_k = 3
event_window_hours = 3
event_min_fog_stations = 80
event_min_regions = 3
event_min_lon_span = 10.0
event_min_lat_span = 4.0
event_gap_hours = 24

# 维度：与 PMST_net_test_11_s2_pm10.py 对齐
window_size = 12
dyn_vars_count = 27
extra_feat_dim = 36

# 输出目录
#output_dir = output_dir or os.path.join(BASE, f"paper_eval_results_{exp_id}_no_season_2")
os.makedirs(output_dir, exist_ok=True)

ckpt_path = os.path.join(ckpt_dir, f"{exp_id}_S2_PhaseB_best_score.pt")

# RobustScaler：与训练时 dyn_vars_count=27（PM2.5+PM10 位于 dyn 末两维）及 log1p 索引一致
scaler_path = os.path.join(ckpt_dir, "robust_scaler_w12_dyn27_s2_48h_pm10.pkl")

season_th_path = os.path.join(ckpt_dir, f"{exp_id}_season_thresholds.pt")

print("BASE:", BASE)
print("Output:", output_dir)
print("Data:", data_dir)
print("IFS:", ifs_nc)
print("ckpt_path:", ckpt_path)
print("scaler_path:", scaler_path)
print("season_th_path:", season_th_path)
print("no_calibration:", no_calibration)
print("use_cpu:", use_cpu)


BASE: /public/home/putianshu/vis_mlp
Output: paper_eval_results_pm10_pm25_11_s2
Data: /public/home/putianshu/vis_mlp/ml_dataset_s2_tianji_12h_pm10_pm25_monthtail_2
IFS: /public/home/putianshu/vis_mlp/VIS_IDW_KDTree_20250101_20251231.nc
ckpt_path: /public/home/putianshu/vis_mlp/checkpoints/exp_1776227576_pm10_more_temp_search_S2_PhaseB_best_score.pt
scaler_path: /public/home/putianshu/vis_mlp/checkpoints/robust_scaler_w12_dyn27_s2_48h_pm10.pkl
season_th_path: /public/home/putianshu/vis_mlp/checkpoints/exp_1776227576_pm10_more_temp_search_season_thresholds.pt
no_calibration: True
use_cpu: False


In [11]:
## 2. Non-torch imports & paper_eval modules

import types
import joblib
import numpy as np
import pandas as pd
import importlib.util
from sklearn.metrics import classification_report

sys.dont_write_bytecode = True
importlib.invalidate_caches()

# 清理旧的 paper_eval 子模块缓存（避免 notebook 长时间运行后混用旧模块）
for _name in list(sys.modules.keys()):
    if _name == "paper_eval" or _name.startswith("paper_eval."):
        del sys.modules[_name]

# 构造一个指向源码目录的轻量 package，使相对导入可正常工作
paper_eval_pkg = types.ModuleType("paper_eval")
paper_eval_pkg.__path__ = [PAPER_EVAL_DIR]
paper_eval_pkg.__file__ = os.path.join(PAPER_EVAL_DIR, "__init__.py")
sys.modules["paper_eval"] = paper_eval_pkg


def _load_paper_eval_submodule(submodule_name):
    full_name = f"paper_eval.{submodule_name}"
    module_path = os.path.join(PAPER_EVAL_DIR, f"{submodule_name}.py")
    spec = importlib.util.spec_from_file_location(full_name, module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[full_name] = module
    spec.loader.exec_module(module)
    return module


plot_style_mod = _load_paper_eval_submodule("plot_style")
metrics_core_mod = _load_paper_eval_submodule("metrics_core")
plot_classification_mod = _load_paper_eval_submodule("plot_classification")
plot_spatial_mod = _load_paper_eval_submodule("plot_spatial")
plot_scenarios_mod = _load_paper_eval_submodule("plot_scenarios")

pred_from_thresholds = metrics_core_mod.pred_from_thresholds
pred_from_thresholds_mutual = getattr(metrics_core_mod, "pred_from_thresholds_mutual", pred_from_thresholds)
pred_from_joint_thresholds = getattr(metrics_core_mod, "pred_from_joint_thresholds", pred_from_thresholds)
compute_rare_event_report = metrics_core_mod.compute_rare_event_report

if hasattr(metrics_core_mod, "pred_from_season_thresholds"):
    pred_from_season_thresholds = metrics_core_mod.pred_from_season_thresholds
else:
    # 兼容：若 metrics_core 未提供 pred_from_season_thresholds，就在 notebook 内做一个 fallback
    def pred_from_season_thresholds(
        probs,
        months,
        season_thresholds,
        default_fog_th=0.46,
        default_mist_th=0.38,
        rule="default",
    ):
        month_to_season = {
            12: "DJF", 1: "DJF", 2: "DJF",
            3: "MAM", 4: "MAM", 5: "MAM",
            6: "JJA", 7: "JJA", 8: "JJA",
            9: "SON", 10: "SON", 11: "SON",
        }
        months = np.asarray(months, dtype=np.int32).ravel()
        n = probs.shape[0]
        fog_th_arr = np.full(n, default_fog_th, dtype=np.float64)
        mist_th_arr = np.full(n, default_mist_th, dtype=np.float64)
        for i in range(n):
            season = month_to_season.get(int(months[i]))
            if season and season in season_thresholds:
                fog_th_arr[i] = season_thresholds[season]["fog_th"]
                mist_th_arr[i] = season_thresholds[season]["mist_th"]
        if rule == "mutual":
            return pred_from_thresholds_mutual(probs, fog_th_arr, mist_th_arr)
        if rule == "joint":
            return pred_from_joint_thresholds(probs, fog_th_arr, mist_th_arr)
        return metrics_core_mod.pred_from_thresholds(probs, fog_th_arr, mist_th_arr)


setup_paper_style = plot_style_mod.setup_paper_style
save_figure = plot_style_mod.save_figure
plot_confusion_matrix_normalized = plot_classification_mod.plot_confusion_matrix_normalized
plot_per_class_prf1 = plot_classification_mod.plot_per_class_prf1
plot_pr_curves = plot_classification_mod.plot_pr_curves
plot_threshold_sweep = plot_classification_mod.plot_threshold_sweep
plot_reliability_diagram = plot_classification_mod.plot_reliability_diagram
plot_fog_recall_map = plot_spatial_mod.plot_fog_recall_map
plot_fpr_map = plot_spatial_mod.plot_fpr_map

run_widespread_event_evaluation = getattr(plot_spatial_mod, "run_widespread_event_evaluation", None)
plot_scenario_bars = plot_scenarios_mod.plot_scenario_bars
# 与 run_paper_eval.py 一致：按起报 init 分层、按 valid hour 分层
enrich_meta_forecast_init = getattr(plot_scenarios_mod, "enrich_meta_forecast_init", lambda m: m)
save_metrics_by_valid_hour = getattr(plot_scenarios_mod, "save_metrics_by_valid_hour", None)
plot_forecast_init_comparison = getattr(plot_scenarios_mod, "plot_forecast_init_comparison", None)
save_forecast_init_metrics_table = getattr(plot_scenarios_mod, "save_forecast_init_metrics_table", None)

print("paper_eval modules loaded OK.")


paper_eval modules loaded OK.


In [12]:
## 3. Helper functions (PM10+PM2.5 dyn preprocessing)

# 与训练脚本 CONFIG 一致，否则模型与 checkpoint/特征预处理会不匹配
EXTRA_FEAT_DIMS = int(extra_feat_dim)
REGIME_FEAT_DIM = 6  # 仅在模型显式需要 regime 特征时才用到


def load_test_data(data_dir, scaler_path, window_size=12):
    """Load X_test/y_test/meta_test from data_dir."""
    X_path = os.path.join(data_dir, "X_test.npy")
    y_path = os.path.join(data_dir, "y_test.npy")
    meta_path = os.path.join(data_dir, "meta_test.csv")
    for p in [X_path, y_path, meta_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Not found: {p}")

    y_raw = np.load(y_path)
    if np.max(y_raw) < 100:
        y_raw = y_raw * 1000.0

    y_cls = np.zeros(len(y_raw), dtype=np.int64)
    y_cls[y_raw >= 500] = 1
    y_cls[y_raw >= 1000] = 2

    meta = pd.read_csv(meta_path, parse_dates=["time"])
    meta["hour"] = meta["time"].dt.hour
    meta["month"] = meta["time"].dt.month
    meta = enrich_meta_forecast_init(meta)

    scaler = joblib.load(scaler_path) if os.path.exists(scaler_path) else None
    return X_path, y_cls, y_raw, meta, scaler


def _regime_features_from_meta(meta_slice):
    """From meta (month/hour/lat/lon) to 6-dim regime features."""
    month = np.asarray(meta_slice["month"].values, dtype=np.int32)
    hour = np.asarray(meta_slice["hour"].values, dtype=np.int32)
    lats = meta_slice["lat"].values if "lat" in meta_slice.columns else np.zeros(len(month))
    lons = meta_slice["lon"].values if "lon" in meta_slice.columns else np.zeros(len(month))

    is_coastal = ((lons >= 118) & (lats >= 18) & (lats <= 42)).astype(np.float32)
    is_day = ((hour >= 6) & (hour < 18)).astype(np.float32)

    djf = ((month == 12) | (month <= 2)).astype(np.float32)
    mam = ((month >= 3) & (month <= 5)).astype(np.float32)
    jja = ((month >= 6) & (month <= 8)).astype(np.float32)
    son = ((month >= 9) & (month <= 11)).astype(np.float32)

    return np.column_stack([djf, mam, jja, son, is_day, is_coastal]).astype(np.float32)


def run_inference(
    X_path,
    scaler,
    model,
    device,
    batch_size=1024,
    window_size=12,
    temperature=None,
    meta=None,
    dyn_vars_count=26,
):
    """Inference with preprocessing layout matching PMSTDataset in PMST_net_test_11_s2_pm10.py."""
    import torch
    import torch.nn.functional as F

    T = 1.0 if temperature is None or temperature == 1.0 else float(temperature)

    X = np.load(X_path, mmap_mode="r")
    N = len(X)

    split_dyn = int(dyn_vars_count) * window_size  # win_size * dyn_vars

    # 与 PMST_net_test_11_s2_pm10._dyn_indices_log1p 一致：dyn>=27 时对末两维（PM2.5、PM10）做 log1p
    log_mask = np.zeros(split_dyn, dtype=bool)
    dyn_i = int(dyn_vars_count)
    log_idx = [2, 4, 9]
    if dyn_i >= 27:
        log_idx.extend([dyn_i - 2, dyn_i - 1])
    else:
        log_idx.append(dyn_i - 1)
    for t in range(window_size):
        base = t * dyn_i
        for i in log_idx:
            log_mask[base + i] = True

    use_cuda = device.type == "cuda"
    non_blocking = use_cuda

    need_regime = getattr(model, "regime_feat_dim", 0)
    if need_regime and meta is None:
        raise ValueError("model has regime_feat_dim but meta was not passed to run_inference")

    all_probs = []
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)

        rows = X[start:end].astype(np.float32)

        # feats layout: [dyn_flat (split_dyn), static_cont(5)]
        feats = rows[:, : split_dyn + 5]
        feats[:, :split_dyn] = np.where(
            log_mask,
            np.log1p(np.maximum(feats[:, :split_dyn], 0)),
            feats[:, :split_dyn],
        )

        if scaler is not None:
            feats = (feats - scaler.center_) / (scaler.scale_ + 1e-6)

        veg = rows[:, split_dyn + 5 : split_dyn + 6]  # [N,1]
        extra = rows[:, split_dyn + 6 :]  # [N,extra]

        # Guard: ensure extra dims consistent (should match EXTRA_FEAT_DIMS=36)
        if extra.shape[1] < EXTRA_FEAT_DIMS:
            extra = np.pad(extra, ((0, 0), (0, EXTRA_FEAT_DIMS - extra.shape[1])), mode="constant", constant_values=0)
        elif extra.shape[1] > EXTRA_FEAT_DIMS:
            extra = extra[:, :EXTRA_FEAT_DIMS]

        final = np.concatenate([np.clip(feats, -10, 10), veg, np.clip(extra, -10, 10)], axis=1)

        if need_regime and meta is not None:
            regime = _regime_features_from_meta(meta.iloc[start:end])
            final = np.concatenate([final, regime], axis=1)

        final = np.nan_to_num(final, nan=0.0)

        x = torch.from_numpy(final).float().to(device, non_blocking=non_blocking)
        with torch.inference_mode():
            fine, _, _ = model(x)
            probs = F.softmax(fine / T, dim=1)

        all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs, axis=0)


def visibility_boundary_band(y_raw):
    """Visibility bins for diagnostics: <400, 400-600, 600-800, 800-1200, >1200."""
    y_raw = np.asarray(y_raw, dtype=np.float64)
    band = np.full(len(y_raw), ">1200", dtype=object)
    band[y_raw < 400] = "<400"
    band[(y_raw >= 400) & (y_raw < 600)] = "400-600"
    band[(y_raw >= 600) & (y_raw < 800)] = "600-800"
    band[(y_raw >= 800) & (y_raw < 1200)] = "800-1200"
    return band


def export_per_sample_table(meta, y_true, y_raw, pred, probs, output_path):
    """Export per-sample table. 若 meta 含 init_time / lead_hours（per-init 数据集），一并写出。"""
    base_cols = ["station_id", "lat", "lon", "time", "month", "hour"]
    for c in ("init_time", "init_hour", "lead_hours", "lead_hour"):
        if c in meta.columns and c not in base_cols:
            base_cols.append(c)
    df = meta[base_cols].copy()
    df["y_true"] = y_true
    df["y_raw"] = y_raw
    df["pred"] = pred
    df["p_fog"] = probs[:, 0]
    df["p_mist"] = probs[:, 1]
    df["visibility_band"] = visibility_boundary_band(y_raw)
    df.to_csv(output_path, index=False, float_format="%.6f")
    return df


def aggregate_station_metrics(meta, y_true, preds):
    """Station-level metrics used for spatial maps."""
    stations = meta[["station_id", "lat", "lon"]].drop_duplicates("station_id")
    rows = []
    for _, row in stations.iterrows():
        sid = row["station_id"]
        m = (meta["station_id"] == sid).values
        y_s = y_true[m]
        p_s = preds[m]

        n_fog_true = (y_s == 0).sum()
        n_mist_true = (y_s == 1).sum()
        n_clear_true = (y_s == 2).sum()

        fog_recall = ((y_s == 0) & (p_s == 0)).sum() / n_fog_true if n_fog_true > 0 else 0.0
        fog_pred = (p_s == 0)
        fog_precision = ((y_s == 0) & fog_pred).sum() / fog_pred.sum() if fog_pred.sum() > 0 else 0.0

        mist_recall = ((y_s == 1) & (p_s == 1)).sum() / n_mist_true if n_mist_true > 0 else 0.0
        mist_pred = (p_s == 1)
        mist_precision = ((y_s == 1) & mist_pred).sum() / mist_pred.sum() if mist_pred.sum() > 0 else 0.0

        # False positive rate for low-vis predictions on true Clear
        fpr_val = ((p_s <= 1) & (y_s == 2)).sum() / n_clear_true if n_clear_true > 0 else 0.0

        rows.append({
            "station_id": sid,
            "lat": row["lat"],
            "lon": row["lon"],
            "fog_recall": fog_recall,
            "fog_precision": fog_precision,
            "mist_recall": mist_recall,
            "mist_precision": mist_precision,
            "fpr_fog": fpr_val,
            "overall_acc": float((p_s == y_s).mean()),
            "n_fog": int(n_fog_true),
            "n_mist": int(n_mist_true),
            "n_clear": int(n_clear_true),
            "n_total": int(len(y_s)),
        })

    return pd.DataFrame(rows)


# Scenario slicing helpers (same fallback logic as run_paper_eval)
if hasattr(plot_scenarios_mod, "derive_scenario_columns"):
    derive_scenario_columns = plot_scenarios_mod.derive_scenario_columns
else:
    def derive_scenario_columns(meta):
        df = meta.copy()
        if "latitude" in df.columns and "lat" not in df.columns:
            df["lat"] = df["latitude"]
        if "longitude" in df.columns and "lon" not in df.columns:
            df["lon"] = df["longitude"]
        if df["time"].dtype == object or "datetime" not in str(df["time"].dtype):
            df["time"] = pd.to_datetime(df["time"])

        df["hour"] = df["time"].dt.hour.values
        df["month"] = df["time"].dt.month.values

        lats = df["lat"].values
        lons = df["lon"].values
        df["is_coastal"] = ((lons >= 118) & (lats >= 18) & (lats <= 42)).astype(int)

        region = np.full(len(df), "Other", dtype=object)
        region_defs = [
            ("Northeast", 38.5, 54, 118, 136),
            ("North_China", 32, 42.5, 110, 120),
            ("East_China", 27, 35, 115, 123),
            ("Central_China", 26, 34, 108, 118),
            ("South_China", 18, 26, 105, 120),
            ("Southwest", 21, 35, 97, 108),
            ("Northwest", 31, 50, 73, 111),
        ]
        for name, lat_min, lat_max, lon_min, lon_max in region_defs:
            mask = (lats >= lat_min) & (lats <= lat_max) & (lons >= lon_min) & (lons <= lon_max)
            region[mask] = name
        df["region"] = region
        return df


if hasattr(plot_scenarios_mod, "build_confusion_summaries_and_bottleneck_table"):
    build_confusion_summaries_and_bottleneck_table = plot_scenarios_mod.build_confusion_summaries_and_bottleneck_table
else:
    def build_confusion_summaries_and_bottleneck_table(eval_df, output_dir):
        def _confusion_type(y_true, pred):
            if y_true == pred:
                return "correct"
            if y_true == 0 and pred == 1:
                return "fog->mist"
            if y_true == 1 and pred == 0:
                return "mist->fog"
            if y_true == 1 and pred == 2:
                return "mist->clear"
            if y_true == 2 and pred == 1:
                return "clear->mist"
            if y_true == 2 and pred == 0:
                return "clear->fog"
            return "other"

        df = eval_df.copy()
        df["confusion_type"] = [_confusion_type(int(df["y_true"].iloc[i]), int(df["pred"].iloc[i])) for i in range(len(df))]

        season_map = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM", 6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
        df["season"] = df["month"].map(season_map)
        df["day_night"] = df["hour"].map(lambda h: "day" if 6 <= int(h) < 18 else "night")

        if "is_coastal" in df.columns:
            df["coastal_inland"] = df["is_coastal"].map(lambda x: "coastal" if x == 1 else "inland")

        confusion_types = ["fog->mist", "mist->fog", "mist->clear", "clear->mist", "clear->fog"]
        strata = ["season", "day_night", "coastal_inland", "region", "visibility_band"]
        rows_summary = []

        for col in strata:
            if col not in df.columns:
                continue
            for slice_val in df[col].dropna().unique():
                sub = df[df[col] == slice_val]
                n_s = len(sub)
                for ct in confusion_types:
                    cnt = int((sub["confusion_type"] == ct).sum())
                    n_err_s = int((sub["confusion_type"] != "correct").sum())
                    rows_summary.append({
                        "slice_dim": col,
                        "slice_value": slice_val,
                        "confusion_type": ct,
                        "count": cnt,
                        "n_slice": n_s,
                        "rate_over_slice": round(cnt / n_s, 6) if n_s > 0 else 0.0,
                        "rate_among_errors_slice": round(cnt / n_err_s, 6) if n_err_s > 0 else 0.0,
                    })

        pd.DataFrame(rows_summary).to_csv(os.path.join(output_dir, "confusion_summaries_stratified.csv"), index=False)

        n_total = len(df)
        jja_mask = df["season"] == "JJA"
        boundary_bands = ["400-600", "600-800", "800-1200"]
        bnd_mask = df["visibility_band"].isin(boundary_bands) if "visibility_band" in df.columns else np.zeros(n_total, dtype=bool)

        rows = []
        for ct in confusion_types:
            full_count = int((df["confusion_type"] == ct).sum())
            jja_count = int((df.loc[jja_mask, "confusion_type"] == ct).sum())
            bnd_count = int((df.loc[bnd_mask, "confusion_type"] == ct).sum())

            rows.extend([
                {"confusion_type": ct, "scope": "full", "count": full_count, "rate": round(full_count / n_total, 6) if n_total > 0 else 0.0, "n_scope": n_total},
                {"confusion_type": ct, "scope": "JJA", "count": jja_count, "rate": round(jja_count / int(jja_mask.sum()), 6) if int(jja_mask.sum()) > 0 else 0.0, "n_scope": int(jja_mask.sum())},
                {"confusion_type": ct, "scope": "boundary_zone", "count": bnd_count, "rate": round(bnd_count / int(bnd_mask.sum()), 6) if int(bnd_mask.sum()) > 0 else 0.0, "n_scope": int(bnd_mask.sum())},
            ])

        bottleneck_df = pd.DataFrame(rows)
        full_rank = bottleneck_df[bottleneck_df["scope"] == "full"].sort_values("count", ascending=False)
        order = full_rank["confusion_type"].tolist()
        bottleneck_df["rank_full_count"] = bottleneck_df["confusion_type"].map({c: i + 1 for i, c in enumerate(order)})
        bottleneck_df.to_csv(os.path.join(output_dir, "bottleneck_table.csv"), index=False)


print("Helper functions defined (pm10_pm25).")


Helper functions defined (pm10_pm25).


In [13]:
## 4. Load PyTorch model & calibration artifacts

import torch
import torch.nn.functional as F

if use_cpu or os.environ.get("CUDA_VISIBLE_DEVICES") == "":
    device = torch.device("cpu")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from PMST_net_test_11_s2_pm10 import ImprovedDualStreamPMSTNet

model = ImprovedDualStreamPMSTNet(
    dyn_vars_count=dyn_vars_count,
    window_size=window_size,
    hidden_dim=512,
    dropout=0.3,
    extra_feat_dim=extra_feat_dim,
    num_classes=3,
).to(device)

print("Imported model module:", ImprovedDualStreamPMSTNet.__module__)
print("Model temporal input dim:", model.temporal_input_proj.in_features)

state = torch.load(ckpt_path, map_location=device)
state = {k.replace("module.", ""): v for k, v in state.items()}

if "temporal_input_proj.weight" in state:
    ckpt_in = state["temporal_input_proj.weight"].shape[1]
    print("Checkpoint temporal input dim:", ckpt_in)
    if ckpt_in != model.temporal_input_proj.in_features:
        print("  [ERROR] temporal_input_proj.in_features mismatch.")

model.load_state_dict(state, strict=False)
model.eval()

if device.type == "cuda" and torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
    print("Model wrapped with DataParallel, GPUs:", torch.cuda.device_count())

print("Loaded checkpoint:", ckpt_path)

# ---- Calibration artifacts ----
season_thresholds = None
temperature = None

if (not no_calibration) and os.path.exists(season_th_path):
    try:
        try:
            cal = torch.load(season_th_path, map_location="cpu", weights_only=True)
        except TypeError:
            cal = torch.load(season_th_path, map_location="cpu")

        season_thresholds = cal.get("season_thresholds") or None
        temperature = cal.get("temperature")
        if temperature is not None:
            temperature = float(temperature)

        print(
            "Calibration loaded:",
            season_th_path,
            "| temperature:",
            temperature,
            "| seasons:",
            list(season_thresholds.keys()) if season_thresholds else [],
        )
    except Exception as e:
        print("[WARN] Could not load calibration:", e)

if season_thresholds is None:
    print("Using global thresholds (no_calibration=True or calibration artifact missing).")


Imported model module: PMST_net_test_11_s2_pm10
Model temporal input dim: 14
Checkpoint temporal input dim: 14
Model wrapped with DataParallel, GPUs: 4
Loaded checkpoint: /public/home/putianshu/vis_mlp/checkpoints/exp_1776227576_pm10_more_temp_search_S2_PhaseB_best_score.pt
Using global thresholds (no_calibration=True or calibration artifact missing).


In [14]:
## 5. Load data, run inference, and threshold prediction

X_path, y_cls, y_raw, meta, scaler = load_test_data(data_dir, scaler_path, window_size=window_size)
print(f"Test samples: {len(y_cls)}, Stations: {meta['station_id'].nunique()}")

probs = run_inference(
    X_path,
    scaler,
    model,
    device,
    batch_size=batch_size,
    window_size=window_size,
    temperature=temperature,
    meta=meta,
    dyn_vars_count=dyn_vars_count,
)

rule = threshold_rule

if season_thresholds is not None:
    preds = pred_from_season_thresholds(
        probs,
        meta["month"].values,
        season_thresholds,
        fog_th,
        mist_th,
        rule=rule,
    )
else:
    if rule == "mutual":
        preds = pred_from_thresholds_mutual(probs, fog_th, mist_th)
    elif rule == "joint":
        preds = pred_from_joint_thresholds(probs, fog_th, mist_th)
    else:
        preds = pred_from_thresholds(probs, fog_th, mist_th)

report = compute_rare_event_report(
    probs,
    y_cls,
    fog_th,
    mist_th,
    pred=preds,
)

print("Rare-event metrics:")
for k, v in report.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.4f}")


Test samples: 1730226, Stations: 2167
Rare-event metrics:
  Fog_P: 0.1878
  Fog_R: 0.4173
  Fog_F1: 0.2590
  Mist_P: 0.0857
  Mist_R: 0.3911
  Mist_F1: 0.1406
  Clear_P: 0.9925
  Clear_R: 0.9474
  Clear_F1: 0.9694
  Fog_CSI: 0.1488
  Fog_POD: 0.4173
  Fog_FAR: 0.8122
  Fog_PSS: -0.3949
  Fog_HSS: 0.2466
  Mist_CSI: 0.0756
  Mist_POD: 0.3911
  Mist_FAR: 0.9143
  Mist_PSS: -0.5231
  Mist_HSS: 0.1286
  low_vis_precision: 0.2043
  false_positive_rate: 0.0526
  macro_f1: 0.4563
  balanced_acc: 0.5853
  mcc: 0.0000
  Brier_Fog: 0.0147
  Brier_Mist: 0.0227
  ECE: 0.0139


In [15]:
## 6. Save reports, per-sample table, and diagnostics

with open(os.path.join(output_dir, "rare_event_report.txt"), "w") as f:
    f.write(
        classification_report(
            y_cls,
            preds,
            target_names=["Fog", "Mist", "Clear"],
        )
    )
    f.write("\n\nRare-event metrics:\n")
    for k, v in report.items():
        if isinstance(v, (int, float)):
            f.write(f"  {k}: {v:.4f}\n")

sta_df = aggregate_station_metrics(meta, y_cls, preds)
sta_df.to_csv(os.path.join(output_dir, "station_metrics.csv"), index=False)
print(f"Saved: rare_event_report.txt, station_metrics.csv ({len(sta_df)} stations).")

per_sample_path = os.path.join(output_dir, "per_sample_eval.csv")
eval_df = export_per_sample_table(meta, y_cls, y_raw, preds, probs, per_sample_path)
print(f"Saved: per_sample_eval.csv ({len(eval_df)} rows) -> {per_sample_path}")

eval_df = derive_scenario_columns(eval_df)
build_confusion_summaries_and_bottleneck_table(eval_df, output_dir)
print("Saved diagnostics: confusion_summaries_stratified.csv, bottleneck_table.csv (if enabled).")


Saved: rare_event_report.txt, station_metrics.csv (2167 stations).
Saved: per_sample_eval.csv (1730226 rows) -> paper_eval_results_pm10_pm25_11_s2/per_sample_eval.csv
  [Diagnostics] Stratified confusion summaries -> paper_eval_results_pm10_pm25_11_s2/confusion_summaries_stratified.csv
  [Diagnostics] Bottleneck table (JJA + boundary-zone) -> paper_eval_results_pm10_pm25_11_s2/bottleneck_table.csv
Saved diagnostics: confusion_summaries_stratified.csv, bottleneck_table.csv (if enabled).


In [16]:
## 7. Generate figures (confusion/PR/reliability/maps/scenarios/events)

setup_paper_style()
class_names = ["Fog", "Mist", "Clear"]

plot_confusion_matrix_normalized(
    y_cls,
    preds,
    class_names,
    os.path.join(output_dir, "fig3_confusion_matrix.png"),
)

plot_per_class_prf1(report, os.path.join(output_dir, "fig3_prf1_bars.png"))
plot_pr_curves(probs, y_cls, class_names, os.path.join(output_dir, "fig4_pr_curves.png"))
plot_threshold_sweep(
    probs,
    y_cls,
    os.path.join(output_dir, "fig4_threshold_sweep.png"),
    fog_th,
    mist_th,
)
plot_reliability_diagram(probs, y_cls, os.path.join(output_dir, "fig5_reliability.png"))

# Station-level spatial maps
if os.path.exists(shp_path):
    plot_fog_recall_map(
        sta_df,
        os.path.join(output_dir, "fig8_station_fog_recall.png"),
        shp_path=shp_path,
        min_fog_events=5,
    )
    plot_fpr_map(
        sta_df,
        os.path.join(output_dir, "fig8_station_fpr.png"),
        shp_path=shp_path,
        min_clear_events=20,
    )
else:
    print("Shapefile not found, skipping fog/fpr maps.")

# Scenario robustness bars（含 Forecast_init_00UTC / 12UTC，见 fig7_scenario_robustness_scenario_table.csv）
meta_full = meta.copy()
meta_full["y_true"] = y_cls
meta_full["pred"] = preds
meta_full["p_fog"] = probs[:, 0]
meta_full["p_mist"] = probs[:, 1]
scenario_results = plot_scenario_bars(
    meta_full,
    os.path.join(output_dir, "fig7_scenario_robustness.png"),
    fog_th=fog_th,
    mist_th=mist_th,
    threshold_rule=threshold_rule,
)
# 与 run_paper_eval.py 对齐：按起报时分层 CSV + 图；按 valid hour 分层 CSV（需 meta 含 init_time 才有起报分层）
if save_forecast_init_metrics_table is not None and scenario_results:
    _p = os.path.join(output_dir, "metrics_by_forecast_init.csv")
    save_forecast_init_metrics_table(scenario_results, _p)
    print("[Init] saved", _p)
if plot_forecast_init_comparison is not None and scenario_results:
    _p = os.path.join(output_dir, "fig7b_forecast_init_comparison.png")
    _fig = plot_forecast_init_comparison(scenario_results, _p)
    if _fig is not None:
        print("[Init] saved", _p)
if save_metrics_by_valid_hour is not None:
    meta_h = derive_scenario_columns(meta_full)
    _p = os.path.join(output_dir, "metrics_by_valid_hour.csv")
    save_metrics_by_valid_hour(y_cls, preds, meta_h, _p)
    print("[Valid hour] saved", _p)

# Optional widespread event evaluation
if run_widespread_event_evaluation is not None:
    try:
        event_df = run_widespread_event_evaluation(
            meta=meta,
            y_true=y_cls,
            y_true_raw=y_raw,
            pmst_pred=preds,
            output_dir=output_dir,
            shp_path=shp_path,
            ifs_nc_path=ifs_nc,
            top_k=event_top_k,
            window_hours=event_window_hours,
            min_fog_stations=event_min_fog_stations,
            min_regions=event_min_regions,
            min_lon_span=event_min_lon_span,
            min_lat_span=event_min_lat_span,
            gap_hours=event_gap_hours,
        )
        print(f"Widespread fog-event evaluation saved for {len(event_df)} events.")
    except Exception as e:
        print(f"[WARN] Event evaluation skipped: {e}")
else:
    print("Skipping widespread event evaluation (run_widespread_event_evaluation not available).")

print("All figures/scenario outputs saved to", output_dir)


  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig3_confusion_matrix.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig3_prf1_bars.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig4_pr_curves.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig4_threshold_sweep.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig5_reliability.png
  [Map] Masked 1166 stations with n_fog<5
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig8_station_fog_recall.png
  [Map] Masked 1 stations with n_clear<20
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig8_station_fpr.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig7_scenario_robustness.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig7_scenario_robustness_detailed.png
[Init] saved paper_eval_results_pm10_pm25_11_s2/metrics_by_forecast_init.csv
[Valid hour] saved paper_eval_results_pm10_pm25_11_s2/metrics_by_valid_hour.csv
  [IFS] Matched 1593051/1730226 samples from VIS_IDW_KDTree_20250101_20251

In [17]:
## 8. IFS matched comparison (PMST-vs-IFS) + Mist map

load_ifs_baseline = getattr(plot_spatial_mod, "load_ifs_baseline", None)
plot_mist_recall_map = getattr(plot_spatial_mod, "plot_mist_recall_map", None)

if load_ifs_baseline is None:
    print("IFS baseline loader not found in plot_spatial; skipping IFS comparison.")
else:
    ifs_pred = np.full(len(y_cls), -1, dtype=np.int64)
    ifs_valid = np.zeros(len(y_cls), dtype=bool)

    report_matched = None
    report_ifs = None

    try:
        ifs_pred, ifs_vis_raw, ifs_valid = load_ifs_baseline(meta, ifs_nc)
        n_ifs_valid = int(ifs_valid.sum())
        print(f"[IFS] Matched {n_ifs_valid}/{len(y_cls)} samples from IFS baseline.")

        if n_ifs_valid > 0:
            report_matched = compute_rare_event_report(
                probs[ifs_valid],
                y_cls[ifs_valid],
                fog_th,
                mist_th,
                pred=preds[ifs_valid],
            )
            report_ifs = compute_rare_event_report(
                np.zeros((n_ifs_valid, probs.shape[1]), dtype=np.float64),
                y_cls[ifs_valid],
                fog_th,
                mist_th,
                pred=ifs_pred[ifs_valid],
            )

            # Overwrite fig3 with matched PMST-vs-IFS comparison
            plot_confusion_matrix_normalized(
                y_cls[ifs_valid],
                preds[ifs_valid],
                ["Fog", "Mist", "Clear"],
                os.path.join(output_dir, "fig3_confusion_matrix.png"),
                baseline_pred=ifs_pred[ifs_valid],
            )
            plot_per_class_prf1(
                report_matched,
                os.path.join(output_dir, "fig3_prf1_bars.png"),
                baseline_stats=report_ifs,
            )
        else:
            print("[WARN] IFS baseline matched 0 samples; keep PMST-only fig3 outputs.")
    except Exception as e:
        print(f"[WARN] IFS baseline load skipped: {e}")

# Mist recall map
if plot_mist_recall_map is not None:
    if os.path.exists(shp_path):
        plot_mist_recall_map(
            sta_df,
            os.path.join(output_dir, "fig8_station_mist_recall.png"),
            shp_path=shp_path,
            min_mist_events=5,
        )
    else:
        print("Shapefile not found, skipping mist recall map.")
else:
    print("plot_mist_recall_map not found; skipping mist recall map.")


  [IFS] Matched 1593051/1730226 samples from VIS_IDW_KDTree_20250101_20251231.nc
[IFS] Matched 1593051/1730226 samples from IFS baseline.
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig3_confusion_matrix.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig3_prf1_bars.png
  [Map] Masked 1182 stations with n_mist<5
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig8_station_mist_recall.png


In [18]:
# --- 任务 1：分时 / 分季 / 分地区 三张独立指标图（基于 scenario_results，与 fig7 同源） ---
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import seaborn as sns

eval_scenario_detailed = getattr(plot_scenarios_mod, "eval_scenario_detailed", None)
_sr = globals().get("scenario_results", None)
if _sr is None and eval_scenario_detailed is not None:
    meta_full = meta.copy()
    meta_full["y_true"] = y_cls
    meta_full["pred"] = preds
    meta_full["p_fog"] = probs[:, 0]
    meta_full["p_mist"] = probs[:, 1]
    _probs3 = np.column_stack(
        [meta_full["p_fog"].values, meta_full["p_mist"].values,
         1.0 - meta_full["p_fog"].values - meta_full["p_mist"].values]
    )
    _sr, _ = eval_scenario_detailed(
        meta_full, meta_full["y_true"].values, _probs3,
        fog_th, mist_th, threshold_rule=threshold_rule, pred=meta_full["pred"].values,
    )
scenario_results = _sr

BAR_KEYS = ["fog_csi", "fog_pod", "mist_csi", "lv_precision", "lv_recall", "fog_f1", "mist_f1"]
BAR_LABELS = ["Fog CSI", "Fog POD", "Mist CSI", "Low-vis prec.", "Low-vis recall", "Fog F1", "Mist F1"]
LINE_SPECS = [
    ("fog_hss", "Fog HSS", "o", "-"),
    ("fog_tss", "Fog TSS (POD−FPR)", "s", "--"),
    ("mist_hss", "Mist HSS", "^", "-"),
    ("mist_tss", "Mist TSS (POD−FPR)", "v", "--"),
]
MIN_N = 50


def _plot_stratified_bars(results, ordered_names, title, save_path, min_n=MIN_N):
    """Bars on left [0,1]: CSI/POD/F1-style; lines on right [−1,1]: HSS & TSS (same binarization as CSI)."""
    setup_paper_style()
    names = [n for n in ordered_names if n in results and int(results[n].get("n", 0)) >= min_n]
    if not names:
        print(f"[Task1 skip] {title}: no bins with n>={min_n}")
        return
    x = np.arange(len(names), dtype=float)
    n_bar = len(BAR_KEYS)
    w = 0.78 / max(n_bar, 1)
    fig, ax = plt.subplots(figsize=(max(6.0, len(names) * 1.08), 4.6), constrained_layout=True)
    pal = sns.color_palette("colorblind", n_colors=n_bar)
    for i, (mk, mlab) in enumerate(zip(BAR_KEYS, BAR_LABELS)):
        vals = []
        for n in names:
            v = results[n].get(mk, np.nan)
            vals.append(float(v) if v is not None and np.isfinite(v) else 0.0)
        off = (i - (n_bar - 1) / 2.0) * w
        ax.bar(x + off, vals, w * 0.92, label=mlab, color=pal[i], edgecolor="white", linewidth=0.35)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score (0–1)", fontsize=10)
    ax.set_xticks(x)
    lbl = [n.replace("Region_", "").replace(" (", "\n(") for n in names]
    ax.set_xticklabels(lbl, fontsize=9)
    ax.grid(axis="y", alpha=0.35)

    ax2 = ax.twinx()
    pal2 = sns.color_palette("dark", n_colors=len(LINE_SPECS))
    for idx, (mk, mlab, marker, ls) in enumerate(LINE_SPECS):
        ys = []
        for n in names:
            v = results[n].get(mk, np.nan)
            ys.append(float(v) if v is not None and np.isfinite(v) else np.nan)
        ax2.plot(
            x, ys, color=pal2[idx % len(pal2)], marker=marker, ls=ls, lw=1.9, ms=5,
            label=mlab, clip_on=False,
        )
    ax2.set_ylim(-1.05, 1.05)
    ax2.set_ylabel("HSS / TSS (−1–1)", fontsize=10)
    ax2.axhline(0.0, color="#b0b0b0", lw=0.9, ls=":")

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    fig.legend(
        h1 + h2, l1 + l2, ncol=3, fontsize=7.2, frameon=True,
        loc="upper center", bbox_to_anchor=(0.5, 1.18),
    )
    ax.set_title(title, fontsize=11, fontweight="bold", pad=28)
    save_figure(fig, save_path)
    plt.close(fig)


time_order = [
    "Midnight (00–06h)", "Morning (06–12h)", "Afternoon (12–18h)", "Evening (18–24h)",
]
season_order = [
    "DJF (Dec–Feb)", "MAM (Mar–May)", "JJA (Jun–Aug)", "SON (Sep–Nov)",
]
region_order = sorted(
    [k for k in scenario_results if k.startswith("Region_")],
    key=lambda s: s.replace("Region_", ""),
)

_plot_stratified_bars(
    scenario_results, time_order,
    "Metrics by time of day (valid local hour)",
    os.path.join(output_dir, "fig7_split_time_of_day.png"),
)
_plot_stratified_bars(
    scenario_results, season_order,
    "Metrics by season",
    os.path.join(output_dir, "fig7_split_season.png"),
)
_plot_stratified_bars(
    scenario_results, region_order,
    "Metrics by region",
    os.path.join(output_dir, "fig7_split_region.png"),
)
print("[Task1] Saved fig7_split_time_of_day.png, fig7_split_season.png, fig7_split_region.png")

# --- 任务 2：3×3 峰值空间分布（与 fig9 事件面板一致：观测能见度等级 / PMST 等级 / IFS 等级；CLASS_CMAP） ---
_draw_ev_basemap = getattr(plot_spatial_mod, "_draw_event_basemap", None)
CLASS_CMAP = plot_spatial_mod.CLASS_CMAP
CLASS_NORM = plot_spatial_mod.CLASS_NORM
CLASS_COLORS = plot_spatial_mod.CLASS_COLORS
CLASS_SHORT_NAMES = getattr(plot_spatial_mod, "CLASS_SHORT_NAMES", ["Fog", "Mist", "Clear"])
shp_gdf_ev = None
if os.path.exists(shp_path):
    try:
        import geopandas as gpd
        shp_gdf_ev = gpd.read_file(shp_path)
    except Exception:
        shp_gdf_ev = None

ev_csv = os.path.join(output_dir, "event_case_summary.csv")
if not os.path.exists(ev_csv):
    print("[Task2 skip] event_case_summary.csv not found; run event evaluation first.")
else:
    event_df_plot = pd.read_csv(ev_csv, parse_dates=["peak_time"]).head(3)
    if event_df_plot.empty:
        print("[Task2 skip] event_case_summary.csv has no rows.")
    else:
        from matplotlib.patches import Patch

        times_m = pd.to_datetime(meta["time"])
        lon_a = meta["lon"].to_numpy(dtype=float)
        lat_a = meta["lat"].to_numpy(dtype=float)
        y_cls_a = np.asarray(y_cls, dtype=np.int64)
        pred_a = np.asarray(preds, dtype=np.int64)

        try:
            ifs_pred_a = np.asarray(ifs_pred, dtype=np.int64)
            ifs_ok = np.asarray(ifs_valid, dtype=bool)
        except NameError:
            ifs_pred_a = np.full(len(meta), -1, dtype=np.int64)
            ifs_ok = np.zeros(len(meta), dtype=bool)
            _loader = getattr(plot_spatial_mod, "load_ifs_baseline", None)
            if _loader is not None and os.path.exists(ifs_nc):
                try:
                    ifs_pred_a, _, ifs_ok = _loader(meta, ifs_nc)
                except Exception as ex:
                    print("[Task2 warn] IFS reload failed:", ex)

        fig, axes = plt.subplots(3, 3, figsize=(11.2, 9.2), constrained_layout=True)

        for j, (_, er) in enumerate(event_df_plot.iterrows()):
            pk = pd.Timestamp(er["peak_time"])
            msk = (times_m == pk).to_numpy()
            rank = int(er.get("event_rank", j + 1))
            col_title = f"Event {rank} peak\n{pk:%Y-%m-%d %H:00} UTC"

            # Row 0 — 观测能见度等级（Fog / Mist / Clear，与训练标签一致）
            ax = axes[0, j]
            if _draw_ev_basemap is not None:
                _draw_ev_basemap(ax, shp_gdf_ev)
            if msk.any():
                ax.scatter(
                    lon_a[msk], lat_a[msk],
                    c=y_cls_a[msk].astype(float),
                    cmap=CLASS_CMAP, norm=CLASS_NORM,
                    s=14, linewidths=0.06, edgecolors="white", alpha=0.95, zorder=3,
                )
                counts = np.bincount(y_cls_a[msk], minlength=3)
                ax.text(
                    0.02, 0.03, f"F={counts[0]} M={counts[1]} C={counts[2]}",
                    transform=ax.transAxes, fontsize=6.5,
                    bbox=dict(facecolor="white", alpha=0.78, edgecolor="none", pad=1.5),
                )
            ax.set_title(col_title, fontsize=9)
            if j == 0:
                ax.set_ylabel("Observation\n(visibility class)", fontsize=9)

            # Row 1 — PMST 预报等级
            ax = axes[1, j]
            if _draw_ev_basemap is not None:
                _draw_ev_basemap(ax, shp_gdf_ev)
            if msk.any():
                ax.scatter(
                    lon_a[msk], lat_a[msk],
                    c=pred_a[msk].astype(float),
                    cmap=CLASS_CMAP, norm=CLASS_NORM,
                    s=14, linewidths=0.06, edgecolors="white", alpha=0.95, zorder=3,
                )
                counts = np.bincount(pred_a[msk], minlength=3)
                ax.text(
                    0.02, 0.03, f"F={counts[0]} M={counts[1]} C={counts[2]}",
                    transform=ax.transAxes, fontsize=6.5,
                    bbox=dict(facecolor="white", alpha=0.78, edgecolor="none", pad=1.5),
                )
            if j == 0:
                ax.set_ylabel("PMST forecast\n(class)", fontsize=9)

            # Row 2 — IFS 预报等级（无匹配站：灰色）
            ax = axes[2, j]
            if _draw_ev_basemap is not None:
                _draw_ev_basemap(ax, shp_gdf_ev)
            if msk.any():
                unmatched = msk & (~ifs_ok)
                matched = msk & ifs_ok & (ifs_pred_a >= 0)
                if unmatched.any():
                    ax.scatter(
                        lon_a[unmatched], lat_a[unmatched],
                        s=12, c="#D8D8D8", linewidths=0, zorder=2, alpha=0.75,
                    )
                if matched.any():
                    ax.scatter(
                        lon_a[matched], lat_a[matched],
                        c=ifs_pred_a[matched].astype(float),
                        cmap=CLASS_CMAP, norm=CLASS_NORM,
                        s=14, linewidths=0.06, edgecolors="white", alpha=0.95, zorder=3,
                    )
                    counts = np.bincount(ifs_pred_a[matched], minlength=3)
                    ax.text(
                        0.02, 0.03, f"F={counts[0]} M={counts[1]} C={counts[2]}",
                        transform=ax.transAxes, fontsize=6.5,
                        bbox=dict(facecolor="white", alpha=0.78, edgecolor="none", pad=1.5),
                    )
            if j == 0:
                ax.set_ylabel("IFS forecast\n(class)", fontsize=9)

        fig.legend(
            handles=[
                Patch(facecolor=CLASS_COLORS[i], edgecolor="none", label=CLASS_SHORT_NAMES[i])
                for i in range(3)
            ],
            labels=list(CLASS_SHORT_NAMES),
            loc="lower center",
            ncol=3,
            frameon=True,
            bbox_to_anchor=(0.5, -0.02),
        )
        fig.suptitle(
            "Peak-hour spatial distribution: observed visibility class vs PMST vs IFS (widespread events)",
            fontsize=11, fontweight="bold",
        )
        _p2 = os.path.join(output_dir, "fig9_events_peak_grid_3x3.png")
        save_figure(fig, _p2)
        plt.close(fig)
        print("[Task2] Saved", _p2)

# --- 任务 3：1×3 生消过程（对应 fig9 metrics 图中 panel a：站点足迹计数） ---
hourly_paths = [
    os.path.join(output_dir, f"fig9_event_{k}_hourly_metrics.csv") for k in (1, 2, 3)
]
if not all(os.path.exists(p) for p in hourly_paths):
    print("[Task3 skip] hourly_metrics CSV missing; run widespread event block first.")
else:
    setup_paper_style()
    fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.8))
    obs_c = "#1a1a1a"
    pmst_c = sns.color_palette("colorblind")[0]
    ifs_c = "#4d4d4d"

    for j, (rank, hpath) in enumerate(zip((1, 2, 3), hourly_paths)):
        hourly = pd.read_csv(hpath)
        x = hourly["hour_offset"].to_numpy()
        ax = axes[j]
        ax.plot(x, hourly["obs_fog_count"], color=obs_c, marker="o", ls="-", lw=2.0, label="Obs fog count")
        ax.plot(x, hourly["obs_low_vis_count"], color=obs_c, marker="^", ls="--", lw=1.5, label="Obs low-vis count")
        ax.plot(x, hourly["pmst_fog_count"], color=pmst_c, marker="o", ls="-", lw=2.0, label="PMST fog count")
        ax.plot(x, hourly["pmst_low_vis_count"], color=pmst_c, marker="^", ls="--", lw=1.5, label="PMST low-vis count")
        ax.plot(x, hourly["ifs_fog_count"], color=ifs_c, marker="s", ls="-", lw=2.0, label="IFS fog count")
        ax.plot(x, hourly["ifs_low_vis_count"], color=ifs_c, marker="s", ls="--", lw=1.5, label="IFS low-vis count")
        ax.axvline(0.0, color="#888888", lw=1.0, ls="--", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels([f"{int(v):+d}h" for v in x])
        ax.set_xlabel("Hour relative to peak")
        ax.set_title(f"Event {rank}: footprint evolution", fontsize=10, fontweight="bold")
        ax.grid(alpha=0.3)

    axes[0].set_ylabel("Station count")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="lower center", ncol=3, frameon=True,
        bbox_to_anchor=(0.5, -0.14), fontsize=8,
    )
    fig.suptitle(
        "Growth and decay of widespread haze/fog footprints (observation vs forecasts)",
        fontsize=11, fontweight="bold", y=0.98,
    )
    fig.tight_layout(rect=(0, 0.14, 1, 0.92))
    _p3 = os.path.join(output_dir, "fig9_events_footprint_evolution_1x3.png")
    save_figure(fig, _p3)
    plt.close(fig)
    print("[Task3] Saved", _p3)


  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig7_split_time_of_day.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig7_split_season.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig7_split_region.png
[Task1] Saved fig7_split_time_of_day.png, fig7_split_season.png, fig7_split_region.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig9_events_peak_grid_3x3.png
[Task2] Saved paper_eval_results_pm10_pm25_11_s2/fig9_events_peak_grid_3x3.png
  [Fig] Saved → paper_eval_results_pm10_pm25_11_s2/fig9_events_footprint_evolution_1x3.png
[Task3] Saved paper_eval_results_pm10_pm25_11_s2/fig9_events_footprint_evolution_1x3.png
